# Clinical/biomedical entity extraction: LoRA fine-tuning vs. few-shot prompting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rkesseku/clinical-lora-extractor/blob/main/notebooks/train_and_eval_colab.ipynb)

Fine-tunes a small open instruction model with LoRA to extract `{"chemicals": [...], "diseases": [...]}` from
**BC5CDR** PubMed abstracts, then compares it with few-shot prompting of the same base model (and optionally a
hosted API model) on the **same held-out test documents**.

* Dataset: BC5CDR is biomedical *literature*, a public proxy for clinical text. No PHI.
* Runtime: **Runtime > Change runtime type > T4 GPU**. With the default sizes the notebook should take roughly
  30-50 minutes on a free T4; shrink `n_train` / `n_test` in the config cell to go faster.

In [ ]:
# 1. GPU check
import subprocess

import torch

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no nvidia-smi")
assert torch.cuda.is_available(), "No GPU found. Use Runtime > Change runtime type > T4 GPU."
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install the package from GitHub (Colab already ships a CUDA build of torch; it is kept as is).
# The second line force-reinstalls the repo itself, so a reused runtime always gets the latest code.
import sys

if "clinical_lora" in sys.modules:
    raise RuntimeError(
        "An older clinical_lora is already imported in this session. "
        "Use Runtime > Restart session (or Disconnect and delete runtime), then Run all again."
    )
%pip install -q "git+https://github.com/rkesseku/clinical-lora-extractor.git@main"
%pip install -q --force-reinstall --no-deps "git+https://github.com/rkesseku/clinical-lora-extractor.git@main"

import clinical_lora

print("clinical_lora", clinical_lora.__version__)

In [ ]:
# 3. Configuration: edit sizes here
from clinical_lora.pipeline import ExperimentConfig

cfg = ExperimentConfig(
    base_model="Qwen/Qwen2.5-1.5B-Instruct",
    n_train=2000,  # training documents, subsampled from the full train split
    n_test=200,  # held-out test documents, identical for every system
    n_shots=3,  # few-shot demonstrations for the prompting baselines
    num_train_epochs=1.0,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_length=1024,
    max_new_tokens=256,
    eval_batch_size=8,
    n_boot=1000,
    load_in_4bit=False,  # True = QLoRA (needs: %pip install -q bitsandbytes)
    out_dir="results",
    adapter_dir="adapters/clinical-lora",
)
cfg

In [ ]:
# 4. Load BC5CDR from the tner/bc5cdr data files (no Hub loading script; clear error otherwise)
from clinical_lora.data import load_bc5cdr, subsample
from clinical_lora.prompts import select_shots

splits = load_bc5cdr()
print({k: len(v) for k, v in splits.items()})
train = subsample(splits["train"], cfg.n_train, cfg.seed)
test = subsample(splits["test"], cfg.n_test, cfg.seed)  # the SAME test set is used for every system
shots = select_shots(splits["train"], cfg.n_shots)
print(test[0].text[:400], "\n->", test[0].target_json())

In [ ]:
# 5. Load the base model and run the few-shot prompting baseline (before any LoRA weights exist)
from clinical_lora.evaluate import SystemRun
from clinical_lora.inference import Generator, load_base_model, load_tokenizer, predict

tokenizer = load_tokenizer(cfg.base_model)
model = load_base_model(cfg.base_model, load_in_4bit=cfg.load_in_4bit)
gen = Generator(model, tokenizer, cfg.max_new_tokens, cfg.eval_batch_size)

runs = []
runs.append(SystemRun(f"base few-shot (k={cfg.n_shots})", predict(gen, test, shots)))
print(runs[-1].predictions[0].raw)

In [ ]:
# 6. (Optional) hosted-API few-shot baseline. Only runs if ANTHROPIC_API_KEY is available.
#    Add it under Colab > Secrets (key icon) as ANTHROPIC_API_KEY, or set it in the environment.
#    Fill in the prices you are billed at (USD per 1M tokens) to get a cost estimate.
import os

try:
    from google.colab import userdata

    os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY"))
except Exception:
    pass

API_PRICE_IN, API_PRICE_OUT = None, None  # e.g. 1.0, 5.0 -- copy from the provider price list
if os.environ.get("ANTHROPIC_API_KEY"):
    from clinical_lora.api_baseline import AnthropicExtractor

    api = AnthropicExtractor()
    price = (
        (API_PRICE_IN, API_PRICE_OUT)
        if API_PRICE_IN is not None and API_PRICE_OUT is not None
        else None
    )
    runs.append(
        SystemRun(
            f"API few-shot ({api.model})",
            api.predict(test, shots),
            price,
            note="hosted model; latency includes network",
        )
    )
else:
    print("No ANTHROPIC_API_KEY: skipping the API baseline.")

In [ ]:
# 7. Train LoRA (loss on the JSON answer tokens only)
from clinical_lora.train import TrainConfig, train_lora

tcfg = TrainConfig(
    output_dir=cfg.adapter_dir,
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    max_length=cfg.max_length,
    seed=cfg.seed,
    logging_steps=10,
)
peft_model, train_metrics = train_lora(model, tokenizer, train, tcfg, quantized=cfg.load_in_4bit)
train_metrics

In [ ]:
# 8. Evaluate the LoRA model on the same test documents (zero-shot: the task is in the adapter)
gen = Generator(peft_model, tokenizer, cfg.max_new_tokens, cfg.eval_batch_size)
runs.append(
    SystemRun(
        "LoRA fine-tuned (0-shot)",
        predict(gen, test),
        extra={
            "train_metrics": {
                k: float(v) for k, v in train_metrics.items() if isinstance(v, (int, float))
            }
        },
    )
)
print(runs[-1].predictions[0].raw)

In [ ]:
# 9. Metrics, report and plots -> results/
from dataclasses import asdict

from IPython.display import Image, Markdown, display

from clinical_lora.evaluate import build_results, save_predictions, write_outputs

meta = {
    "base_model": cfg.base_model,
    "dataset": "BC5CDR (biomedical literature; proxy for clinical text, no PHI)",
    "n_train": len(train),
    "lora": {"r": cfg.lora_r, "alpha": cfg.lora_alpha, "4bit": cfg.load_in_4bit},
    "gpu": torch.cuda.get_device_name(0),
    "config": asdict(cfg),
}
results = build_results(runs, test, meta=meta, n_boot=cfg.n_boot, seed=cfg.seed)
paths = write_outputs(results, cfg.out_dir)
save_predictions(runs, test, cfg.out_dir)
display(Markdown(paths["report"].read_text()))
display(Image(str(paths["f1_by_type"])))
display(Image(str(paths["latency_validity"])))

In [ ]:
# 10. Quick qualitative look at a few test documents (gold vs. LoRA prediction)
lora_run = runs[-1]
for ex, p in list(zip(test, lora_run.predictions))[:3]:
    print("TEXT :", ex.text[:200].replace("\n", " "), "...")
    print("GOLD :", ex.entities)
    print("LORA :", p.entities, "(valid JSON)" if p.parse.valid else "(repaired/invalid)")
    print()

In [ ]:
# 11. (Optional) save the adapter + results to Google Drive
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/clinical-lora-extractor
    !cp -r adapters results /content/drive/MyDrive/clinical-lora-extractor/
    print("Saved to MyDrive/clinical-lora-extractor")

In [ ]:
# 12. Zip results/ for download (then copy the numbers from metrics.json into the README)
import shutil

shutil.make_archive("results", "zip", "results")
try:
    from google.colab import files

    files.download("results.zip")
except Exception:
    print("results.zip written to the working directory")

## Next steps

* Copy the numbers from `results/metrics.json` / `results/report.md` into the README Results section.
* To serve the adapter locally see the README "Serving" section (`CLE_ADAPTER=adapters/clinical-lora`).
* Download the adapter folder (or use step 11) to keep it; Colab runtimes are ephemeral.